In [12]:
import pandas as pd
import numpy as np
import os

In [13]:
SCRIPT_DIR_PATH = os.getcwd()
OUTPUT_POSTPROCESSING_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
SSP_MODELING_DIR_PATH = os.path.dirname(OUTPUT_POSTPROCESSING_DIR_PATH)
SSP_OUTPUT_DIR_PATH = os.path.join(SSP_MODELING_DIR_PATH, "ssp_run_output")
CW_DATA_DIR_PATH = os.path.join(OUTPUT_POSTPROCESSING_DIR_PATH, "data")

In [14]:
ISO3 = "LBY"
REGION_NAME = "libya"
RUN_ID = "sisepuede_results_run_sisepuede_run_2025-11-04T17;58;01.546836"
RUN_DIR_PATH = os.path.join(SSP_OUTPUT_DIR_PATH, RUN_ID)

### Load emission targets and ssp outputs dfs

In [15]:
# Load emission targets
emission_targets_df = pd.read_csv(os.path.join(CW_DATA_DIR_PATH, "emission_targets_libya_2022_base.csv"))
emission_targets_df.head()

,Subsector,Gas,Edgar_Sector,Edgar_Subsector,Edgar_Subsector_Synthetic,Vars,id,LBY,Edgar_Class
0,agrc,CH4,Agriculture,AG - Crops,AG - Crops,emission_co2e_ch4_agrc_anaerobicdom_rice:emiss...,AG - Crops - CH4,0.000293,AG - Crops:CH4
1,agrc,CO2,Agriculture,AG - Crops,AG - Crops,emission_co2e_co2_agrc_biomass_bevs_and_spices...,AG - Crops - CO2,0.015880,AG - Crops:CO2
2,agrc,N2O,Agriculture,AG - Crops,AG - Crops,emission_co2e_n2o_agrc_biomass_burning:emissio...,AG - Crops - N2O,0.759204,AG - Crops:N2O
3,lvst,CH4,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_ch4_lvst_entferm_buffalo:emissio...,AG - Livestock - CH4,0.954793,AG - Livestock:CH4
4,lsmm,CH4,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_ch4_lsmm_anaerobic_digester:emis...,AG - Livestock - CH4,0.954793,AG - Livestock:CH4


In [16]:
# Load output data
ssp_output_df = pd.read_csv(os.path.join(RUN_DIR_PATH, 
                                         f"sisepuede_results_sisepuede_run_2025-11-04T17;58;01.546836_WIDE_INPUTS_OUTPUTS.csv"))
ssp_output_df.head()

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yf_agrc_herbs_and_other_perennial_crops_tonne_ha,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha
0,0,libya,0,0.0,608456.978485,0.0,465682.879988,0.0,111258.160721,388936.117983,...,0.0,0.0,0.0,0.0,0.408792,0.0,0.0,19.854918,10.048107,92.81
1,0,libya,1,0.0,602737.240482,0.0,461305.275390,0.0,110212.289686,385279.963524,...,0.0,0.0,0.0,0.0,0.506511,0.0,0.0,19.501076,10.184352,92.81
2,0,libya,2,0.0,621860.479516,0.0,475941.256804,0.0,113709.030585,397503.865322,...,0.0,0.0,0.0,0.0,0.380297,0.0,0.0,19.386657,10.312220,92.81
3,0,libya,3,0.0,637333.254494,0.0,487783.353564,0.0,116538.273319,407394.327996,...,0.0,0.0,0.0,0.0,0.537927,0.0,0.0,19.576034,10.192840,92.81
4,0,libya,4,0.0,647059.782222,0.0,495227.556859,0.0,118316.797723,413611.691047,...,0.0,0.0,0.0,0.0,0.242343,0.0,0.0,19.486967,10.182148,92.81


### Obtain the ssp output values in the emission targets format

In [17]:
def sum_vars_from_ssp_outputs(
    emission_targets_df: pd.DataFrame,
    ssp_outputs_df: pd.DataFrame,
    vars_col: str = "Vars",
    out_col: str = "ssp_total",
    record_missing_col: str | None = "missing_vars",
    ssp_filter: dict | None = None,
) -> pd.DataFrame:
    """
    For each row in emission_targets_df, split the colon-separated strings in `vars_col`,
    find those columns in ssp_outputs_df, sum their values (over rows & columns), and
    write the total to `out_col` in emission_targets_df.

    Parameters
    ----------
    emission_targets_df : DataFrame
        Must contain a string column `vars_col` with colon-separated names.
    ssp_outputs_df : DataFrame
        Wide table whose columns include the names referenced by `emission_targets_df[vars_col]`.
    vars_col : str
        Column in emission_targets_df with colon-separated variable names.
    out_col : str
        New column to create in emission_targets_df with totals from ssp_outputs_df.
    record_missing_col : str | None
        If provided, creates a column listing any missing vars for each row.
    df2_filter : dict | None
        Optional filters to reduce ssp_outputs_df before summing, e.g.
        {"region": "egypt", "time_period": 7}

    Returns
    -------
    DataFrame
        emission_targets_df with new column `out_col` (and `record_missing_col` if requested).
    """
    # Optionally filter ssp_outputs_df by key=value pairs (e.g., region/time_period)
    if ssp_filter:
        mask = pd.Series(True, index=ssp_outputs_df.index)
        for k, v in ssp_filter.items():
            mask &= (ssp_outputs_df[k] == v)
        ssp_view = ssp_outputs_df.loc[mask]
    else:
        ssp_view = ssp_outputs_df

    # Ensure we only operate on numeric data when summing
    numeric_cols = set(ssp_view.select_dtypes(include=[np.number]).columns)

    def _total_for_vars(vars_str: str):
        if pd.isna(vars_str) or not str(vars_str).strip():
            return np.nan, []

        # Split, strip, and deduplicate while preserving order
        raw = [s.strip() for s in str(vars_str).split(":") if s.strip()]
        seen = set()
        cols = [c for c in raw if not (c in seen or seen.add(c))]

        present = [c for c in cols if c in ssp_view.columns and c in numeric_cols]
        missing = [c for c in cols if c not in ssp_view.columns or c not in numeric_cols]

        if not present or ssp_view.empty:
            return np.nan, missing

        # Sum over all filtered rows & all present columns
        vals = ssp_view[present].to_numpy(dtype=float, copy=False)
        total = np.nansum(vals)
        return float(total), missing

    totals, missings = [], []
    for v in emission_targets_df[vars_col].astype("string"):
        total, missing = _total_for_vars(v)
        totals.append(total)
        missings.append(missing)

    emission_targets_df = emission_targets_df.copy()
    emission_targets_df[out_col] = totals
    if record_missing_col is not None:
        emission_targets_df[record_missing_col] = missings

    return emission_targets_df


# -----------------------------
# Example usage
# -----------------------------

# If DF2 has a single row for the target (e.g., region="egypt", a specific time_period):
# df2_filter = {"region": "egypt"}              # or {"region": "egypt", "time_period": 7}
# If you want to sum across all rows of DF2, set df2_filter = None.

# df1_result = sum_vars_from_df2(DF1, DF2, vars_col="Vars",
#                                out_col="DF2_total",
#                                record_missing_col="Missing_in_DF2",
#                                df2_filter={"region": "egypt"})
# print(df1_result.head())


In [18]:
ssp_output_df.head()

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yf_agrc_herbs_and_other_perennial_crops_tonne_ha,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha
0,0,libya,0,0.0,608456.978485,0.0,465682.879988,0.0,111258.160721,388936.117983,...,0.0,0.0,0.0,0.0,0.408792,0.0,0.0,19.854918,10.048107,92.81
1,0,libya,1,0.0,602737.240482,0.0,461305.275390,0.0,110212.289686,385279.963524,...,0.0,0.0,0.0,0.0,0.506511,0.0,0.0,19.501076,10.184352,92.81
2,0,libya,2,0.0,621860.479516,0.0,475941.256804,0.0,113709.030585,397503.865322,...,0.0,0.0,0.0,0.0,0.380297,0.0,0.0,19.386657,10.312220,92.81
3,0,libya,3,0.0,637333.254494,0.0,487783.353564,0.0,116538.273319,407394.327996,...,0.0,0.0,0.0,0.0,0.537927,0.0,0.0,19.576034,10.192840,92.81
4,0,libya,4,0.0,647059.782222,0.0,495227.556859,0.0,118316.797723,413611.691047,...,0.0,0.0,0.0,0.0,0.242343,0.0,0.0,19.486967,10.182148,92.81


In [19]:
emission_targets_df_extended = sum_vars_from_ssp_outputs(emission_targets_df, ssp_output_df, vars_col="Vars",
                               out_col="ssp_emission",
                               record_missing_col="missing_in_ssp_outputs",
                               ssp_filter={"region": REGION_NAME, "primary_id": 0, "time_period": 7})

emission_targets_df_extended.head()

,Subsector,Gas,Edgar_Sector,Edgar_Subsector,Edgar_Subsector_Synthetic,Vars,id,LBY,Edgar_Class,ssp_emission,missing_in_ssp_outputs
0,agrc,CH4,Agriculture,AG - Crops,AG - Crops,emission_co2e_ch4_agrc_anaerobicdom_rice:emiss...,AG - Crops - CH4,0.000293,AG - Crops:CH4,0.007527,[]
1,agrc,CO2,Agriculture,AG - Crops,AG - Crops,emission_co2e_co2_agrc_biomass_bevs_and_spices...,AG - Crops - CO2,0.015880,AG - Crops:CO2,0.100326,[]
2,agrc,N2O,Agriculture,AG - Crops,AG - Crops,emission_co2e_n2o_agrc_biomass_burning:emissio...,AG - Crops - N2O,0.759204,AG - Crops:N2O,0.053535,[]
3,lvst,CH4,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_ch4_lvst_entferm_buffalo:emissio...,AG - Livestock - CH4,0.954793,AG - Livestock:CH4,3.029978,[]
4,lsmm,CH4,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_ch4_lsmm_anaerobic_digester:emis...,AG - Livestock - CH4,0.954793,AG - Livestock:CH4,0.232867,[]


### Create diff report

In [20]:
# subset the emission targets to create the diff report template
diff_report_df = emission_targets_df_extended[[
    "Subsector",
    "id",
    ISO3,
    "ssp_emission",
]].copy()
diff_report_df.head()

,Subsector,id,LBY,ssp_emission
0,agrc,AG - Crops - CH4,0.000293,0.007527
1,agrc,AG - Crops - CO2,0.015880,0.100326
2,agrc,AG - Crops - N2O,0.759204,0.053535
3,lvst,AG - Livestock - CH4,0.954793,3.029978
4,lsmm,AG - Livestock - CH4,0.954793,0.232867


In [21]:
# merge subsector an id into a single column for clarity
diff_report_df["subsector_id"] = diff_report_df["Subsector"] + " - " + diff_report_df["id"]
diff_report_df = diff_report_df.drop(columns=["Subsector", "id"])
diff_report_df.head()

,LBY,ssp_emission,subsector_id
0,0.000293,0.007527,agrc - AG - Crops - CH4
1,0.015880,0.100326,agrc - AG - Crops - CO2
2,0.759204,0.053535,agrc - AG - Crops - N2O
3,0.954793,3.029978,lvst - AG - Livestock - CH4
4,0.954793,0.232867,lsmm - AG - Livestock - CH4


In [22]:
#rename region column
diff_report_df = diff_report_df.rename(columns={ISO3: "inventory_emission"})
diff_report_df.head()

,inventory_emission,ssp_emission,subsector_id
0,0.000293,0.007527,agrc - AG - Crops - CH4
1,0.015880,0.100326,agrc - AG - Crops - CO2
2,0.759204,0.053535,agrc - AG - Crops - N2O
3,0.954793,3.029978,lvst - AG - Livestock - CH4
4,0.954793,0.232867,lsmm - AG - Livestock - CH4


In [23]:
# Create inventory_share column
diff_report_df["inventory_share"] = diff_report_df["inventory_emission"] / diff_report_df["inventory_emission"].sum()
diff_report_df

,inventory_emission,ssp_emission,subsector_id,inventory_share
0,0.000293,0.007527,agrc - AG - Crops - CH4,0.000003
1,0.015880,0.100326,agrc - AG - Crops - CO2,0.000152
2,0.759204,0.053535,agrc - AG - Crops - N2O,0.007264
3,0.954793,3.029978,lvst - AG - Livestock - CH4,0.009136
4,0.954793,0.232867,lsmm - AG - Livestock - CH4,0.009136
5,0.007607,0.298189,lsmm - AG - Livestock - N2O,0.000073
6,0.000000,0.000000,ccsq - CCSQ - CH4,0.000000
7,0.000000,0.000000,ccsq - CCSQ - CO2,0.000000
8,0.000000,0.000000,ccsq - CCSQ - N2O,0.000000
9,0.087196,0.003781,scoe - EN - Building - CH4,0.000834


In [24]:
# Calculate error column, avoid division by zero by adding a small constant to the denominator
epsilon = 1e-8
diff_report_df["error"] = (diff_report_df["ssp_emission"] - diff_report_df["inventory_emission"]).abs() / (diff_report_df["inventory_emission"] + epsilon)
diff_report_df.head()

,inventory_emission,ssp_emission,subsector_id,inventory_share,error
0,0.000293,0.007527,agrc - AG - Crops - CH4,0.000003,24.692750
1,0.015880,0.100326,agrc - AG - Crops - CO2,0.000152,5.317959
2,0.759204,0.053535,agrc - AG - Crops - N2O,0.007264,0.929485
3,0.954793,3.029978,lvst - AG - Livestock - CH4,0.009136,2.173439
4,0.954793,0.232867,lsmm - AG - Livestock - CH4,0.009136,0.756108


In [25]:
# Set subsector_id at the beginning of the df
diff_report_df = diff_report_df[[
    "subsector_id",
    "inventory_emission",
    "ssp_emission",
    "inventory_share",
    "error",
]]

# sort by error descending
diff_report_df = diff_report_df.sort_values(by="error", ascending=False)
diff_report_df.head(10)

,subsector_id,inventory_emission,ssp_emission,inventory_share,error
37,frst - LULUCF - Forest Land Removals - CO2,0.0,9.161435,0.0,9.161435e+08
45,waso - Waste - Solid Waste - CO2,0.0,1.453752,0.0,1.453752e+08
38,frst - LULUCF - Forest Land Sequestration - CO2,0.0,-0.653036,0.0,6.530362e+07
41,soil - LULUCF - Organic Soil - N2O,0.0,0.502190,0.0,5.021900e+07
40,soil - LULUCF - Organic Soil - CO2,0.0,0.379124,0.0,3.791245e+07
42,lndu - LULUCF - Other Land - CO2,0.0,0.319940,0.0,3.199396e+07
35,lndu - LULUCF - Deforestation - CO2,0.0,0.286009,0.0,2.860095e+07
34,lndu - LULUCF - Deforestation - CH4,0.0,0.033845,0.0,3.384512e+06
36,frst - LULUCF - Forest Land - CH4,0.0,0.007854,0.0,7.853577e+05
46,waso - Waste - Solid Waste - N2O,0.0,0.002536,0.0,2.536456e+05


In [26]:
diff_report_df.tail(40)

,subsector_id,inventory_emission,ssp_emission,inventory_share,error
46,waso - Waste - Solid Waste - N2O,0.000000,0.002536,0.000000,253645.633037
5,lsmm - AG - Livestock - N2O,0.007607,0.298189,0.000073,38.197440
27,ippu - IN - Industrial Processes - CH4,0.003465,0.089878,0.000033,24.938639
0,agrc - AG - Crops - CH4,0.000293,0.007527,0.000003,24.692750
20,inen - EN - Manufacturing/Construction - N2O,0.003152,0.064384,0.000030,19.428604
18,inen - EN - Manufacturing/Construction - CH4,0.001914,0.034974,0.000018,17.274883
28,ippu - IN - Industrial Processes - CO2,2.499458,30.179192,0.023915,11.074293
48,trww - Waste - Wastewater Treatment - N2O,0.103735,0.803921,0.000993,6.749740
1,agrc - AG - Crops - CO2,0.015880,0.100326,0.000152,5.317959
19,inen - EN - Manufacturing/Construction - CO2,2.301802,12.075351,0.022024,4.246042


### Save diff table

In [27]:
diff_report_df.to_csv(os.path.join(RUN_DIR_PATH, f"diff_report_{REGION_NAME}.csv"), index=False)